# Main Work Flow

**1. Simple Conversion**

In [28]:
from docling.document_converter import DocumentConverter

source = r"D:\3. Research Main Room\Publications\1 Current Projects\Data Science and Tourism Research - Conceptual Study\DATA ANALYSIS\Data_files\2024_Data\2024 - 1.pdf"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)

[INFO] 2026-02-11 07:31:02,302 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-11 07:31:02,304 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-11 07:31:02,363 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-11 07:31:02,365 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-11 07:31:03,740 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-11 07:31:03,741 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-11 07:31:03,746 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-11 07:31:03,747 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\r

In [7]:
from IPython.display import Markdown, display

#display(Markdown(result.document.export_to_markdown()))

In [12]:
type(result.document)

docling_core.types.doc.document.DoclingDocument

In [15]:
#dir(result.document)

**2. Custom Conversion**

[Documentation](https://docling-project.github.io/docling/examples/custom_convert/)

**3. Image Annotation**

Its possible but currently not working

In [ ]:


from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    PictureDescriptionVlmEngineOptions,
)
from docling.datamodel.vlm_engine_options import (
    ApiVlmEngineOptions,
    VlmEngineType,
)
from docling.document_converter import DocumentConverter, PdfFormatOption



picture_desc_options = PictureDescriptionVlmEngineOptions.from_preset(
        "granite_vision",
        engine_options=ApiVlmEngineOptions(
            runtime_type=VlmEngineType.API_LMSTUDIO,
            # url is pre-configured for LM Studio (http://localhost:1234/v1/chat/completions)
            # model name is pre-configured from the preset
            timeout=90,
        ),
    )

pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True
pipeline_options.picture_description_options = picture_desc_options
pipeline_options.enable_remote_services = True 

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
        )
    }
)
result = doc_converter.convert(input_doc_path)

for element, _level in result.document.iterate_items():
    if isinstance(element, PictureItem):
        print(
            f"Picture {element.self_ref}\n"
            f"Caption: {element.caption_text(doc=result.document)}\n"
            f"Meta: {element.meta}\n"
        )

**4. Docling Conversion Confidence**

[Documentation](https://docling-project.github.io/docling/concepts/confidence_scores/)

Confidence scores provide a quantitative assessment of document conversion quality. Each confidence report includes a numerical score (0.0 to 1.0) measuring conversion accuracy, and a quality grade (poor, fair, good, excellent) for quick interpretation.

Use cases for confidence grades include:

- Identify documents requiring manual review after the conversion
- Adjust conversion pipelines to the most appropriate for each document type
- Set confidence thresholds for unattended batch conversions
- Catch potential conversion issues early in your workflow.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

# Parse confidence scores
conf = result.confidence

# Create summary table
summary = {
    "Metric": ["Parse", "Layout", "Table", "OCR", "Overall Mean", "Overall Low"],
    "Score": [
        f"{conf.parse_score:.2%}",
        f"{conf.layout_score:.2%}",
        f"{conf.table_score if conf.table_score else 'N/A'}",
        f"{conf.ocr_score:.2%}" if conf.ocr_score else "N/A",
        f"{conf.mean_score:.2%}",
        f"{conf.low_score:.2%}"
    ]
}

df = pd.DataFrame(summary)
display(df)

# Per-page breakdown
print("**Per-Page Quality:**\n")
for page_num, scores in conf.pages.items():
    print(f"Page {page_num}: {scores.mean_grade.value} (layout: {scores.layout_score:.2%})")

,Metric,Score
0,Parse,100.00%
1,Layout,85.23%
2,Table,nan
3,OCR,97.64%
4,Overall Mean,93.11%
5,Overall Low,86.12%


**Per-Page Quality:**

Page 1: good (layout: 73.99%)
Page 2: excellent (layout: 84.28%)
Page 3: excellent (layout: 83.52%)
Page 4: excellent (layout: 87.53%)
Page 5: excellent (layout: 81.21%)
Page 6: excellent (layout: 84.80%)
Page 7: excellent (layout: 88.55%)
Page 8: excellent (layout: 88.38%)
Page 9: excellent (layout: 81.12%)
Page 10: excellent (layout: 94.18%)
Page 11: excellent (layout: 86.28%)
Page 12: excellent (layout: 93.28%)
Page 13: good (layout: 75.64%)
Page 14: excellent (layout: 76.25%)
Page 15: excellent (layout: 93.57%)
Page 16: excellent (layout: 90.39%)
Page 17: excellent (layout: 91.79%)
Page 18: good (layout: 79.36%)


**Model Dump** [Misc]

In [16]:
data_dict = result.document.model_dump()
type(data_dict)


dict

#### 🗒️ **Developer Note: Why model_dump?**

**What is it?**

In Python/Pydantic, model_dump() (or model_dump_json()) is the process of Serialization. It converts a complex, "live" Python Object (which has hidden methods and internal logic) into a "dead" but highly portable Data Structure (Dictionary or JSON String).

**Why am I doing this for Agentic RAG?**

1. **Metadata Preservation:** Unlike export_to_markdown(), which strips away things like bounding boxes and internal IDs, the "dump" keeps the provenance (page numbers and coordinates) intact.

2. **Schema Consistency:** Because Docling is built on Pydantic, the output of this dump is guaranteed to follow a specific structure. This makes it easy to write stable functions to "pluck" out specific tables or excerpts.

3. **Agent Context:** JSON is a "native language" for LLMs. If an Agent needs to see the exact structure of a table or a bibliographic reference, passing it a snippet of the JSON dump is often more accurate than passing raw text.

In [22]:
data_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [ ]:
data_dict['pages']

# Extract **Textual** Content in a Structured Dataframe

In [38]:
def create_chunks_with_metadata(data):
    chunks = []
    
    # Mapping for quick lookup: e.g., {'#/texts/0': {text_object}}
    # This allows us to resolve 'cref' pointers instantly.
    text_map = {f"#/texts/{i}": item for i, item in enumerate(data.get("texts", []))}
    table_map = {f"#/tables/{i}": item for i, item in enumerate(data.get("tables", []))}
    
    # We iterate through the body to follow the document flow
    body_items = data.get("body", {}).get("children", [])
    
    for item in body_items:
        ref = item.get("cref")
        content = ""
        page_no = None
        label = "text"

        # Resolve Text References
        if ref in text_map:
            target = text_map[ref]
            content = target.get("text", "").strip()
            page_no = target.get("prov", [{}])[0].get("page_no") # Prov often holds page info
            label = target.get("label", "text")

        # Resolve Table References (Optional: converting tables to text)
        elif ref in table_map:
            target = table_map[ref]
            # You might want to represent tables as Markdown or CSV strings
            content = f"Table content: {target.get('label', 'unspecified')}" 
            page_no = target.get("prov", [{}])[0].get("page_no")
            label = "table"

        # Only add if there is actual content
        if content:
            chunks.append({
                "content": content,
                "metadata": {
                    "page_number": page_no,
                    "type": label,
                    "source": data.get("name")
                }
            })
            
    return chunks


In [73]:

# Usage
processed_chunks = create_chunks_with_metadata(data_dict)

print(f'total chunks: {len(processed_chunks)}\n')

print(f'Sample Chunk:')
processed_chunks[0]



total chunks: 171

Sample Chunk:


{'content': 'Journal of Hospitality and Tourism Management 61 (2024) 299-312',
 'metadata': {'page_number': 1,
  'type': <DocItemLabel.PAGE_HEADER: 'page_header'>,
  'source': '2024 - 1'}}

**Histogram** of Content Length in Each Chunk

In [ ]:
# Count the Size of 'content' in each chunk
for chunk in processed_chunks:
    chunk['metadata']['content_length'] = len(chunk['content'])

# plot a histogram of content lengths: x-axis chunk number, y-axis content length
import matplotlib.pyplot as plt
content_lengths = [chunk['metadata']['content_length'] for chunk in processed_chunks]
plt.figure(figsize=(10, 5))
plt.bar(range(len(content_lengths)), content_lengths)
plt.xlabel('Chunk Number')
plt.ylabel('Content Length')
plt.title('Content Length of Each Chunk')
plt.show()

View Chunk Content as Scrollable **HTML** Table

In [ ]:
# view chunk_df in a scrollable window (if too large) full lenth column width
from IPython.display import display, HTML
display(HTML(chunk_df.to_html( index=False)))


# **Tables**

In [75]:
data_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [76]:
data_dict['tables']

[{'self_ref': '#/tables/0',
  'parent': {'cref': '#/body'},
  'children': [],
  'content_layer': <ContentLayer.BODY: 'body'>,
  'meta': None,
  'label': <DocItemLabel.TABLE: 'table'>,
  'prov': [{'page_no': 3,
    'bbox': {'l': 126.81782531738281,
     't': 739.4692802429199,
     'r': 468.344482421875,
     'b': 523.9653015136719,
     'coord_origin': <CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>},
    'charspan': (0, 0)}],
  'captions': [],
  'references': [],
  'footnotes': [],
  'image': None,
  'data': {'table_cells': [{'bbox': {'l': 244.33333333333331,
      't': 66.66666666666667,
      'r': 288.66666666666663,
      'b': 91.66666666666666,
      'coord_origin': <CoordOrigin.TOPLEFT: 'TOPLEFT'>},
     'row_span': 1,
     'col_span': 1,
     'start_row_offset_idx': 0,
     'end_row_offset_idx': 1,
     'start_col_offset_idx': 2,
     'end_col_offset_idx': 3,
     'text': 'Specific composition of culture',
     'column_header': True,
     'row_header': False,
     'row_section': False,
 

Think about chunk size should I need to make fix chunk size like 500 words, 512 or 1024

# Lets Test Hybrid Chunker from Docling

In [77]:
type(result.document)

docling_core.types.doc.document.DoclingDocument

In [81]:
from docling.chunking import HybridChunker

doc = result.document

# 2. Use the HybridChunker
# This splits the document but preserves the link to the original PDF
chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2", # Optional: for token-aware splitting
    max_tokens=800, # Optional: max tokens per chunk (adjust based on your needs)
    merge_peers=True # Optional: merge small chunks together
)

chunk_iter = chunker.chunk(doc)

# 3. Process the chunks for your RAG Vector DB
rag_payloads = []

for i, chunk in enumerate(chunk_iter):
    # Extract the provenance (location data)
    # Note: A chunk might span multiple items, so we look at the first one
    page_numbers = set()
    bbox_list = []
    
    if chunk.meta and chunk.meta.doc_items:
        for item in chunk.meta.doc_items:
            # Each item (sentence/paragraph) has provenance
            if hasattr(item, 'prov') and item.prov:
                for p in item.prov:
                    page_numbers.add(p.page_no)
                    bbox_list.append(p.bbox)

    # Create the payload for your Vector DB
    payload = {
        "id": f"chunk_{i}",
        
        # The CONTENT is what the LLM sees (Markdown format)
        "content": chunk.text, 
        
        # The METADATA is what your UI uses to highlight the PDF
        "metadata": {
            "source": source,
            "pages": list(page_numbers), # e.g., [1] or [1, 2] if it crosses a page
            "heading": chunk.meta.headings[0] if chunk.meta.headings else "None" # Context!
        }
    }
    
    rag_payloads.append(payload)

# Preview
print(f"Chunk 1 Content:\n{rag_payloads[0]['content']}")
print(f"Chunk 1 Pages: {rag_payloads[0]['metadata']['pages']}")

Token indices sequence length is longer than the specified maximum sequence length for this model (639 > 512). Running this sequence through the model will result in indexing errors


Chunk 1 Content:
Contents lists available at ScienceDirect
Chunk 1 Pages: [1]


In [7]:
import uuid

def create_research_chunks(data):
    chunks = []

    text_map = {f"#/texts/{i}": item for i, item in enumerate(data.get("texts", []))}
    table_map = {f"#/tables/{i}": item for i, item in enumerate(data.get("tables", []))}
    picture_map = {f"#/pictures/{i}": item for i, item in enumerate(data.get("pictures", []))}

    body_items = data.get("body", {}).get("children", [])

    for item in body_items:
        ref = item.get("cref")

        # ---------- TEXT ----------
        if ref in text_map:
            t = text_map[ref]
            text = t.get("text", "").strip()
            if not text:
                continue

            page_no = t.get("prov", [{}])[0].get("page_no")

            chunks.append({
                "chunk_id": uuid.uuid4().hex,
                "text": text,
                "page": page_no,
                "modality": "text",
                "docling_label": str(t.get("label")),
                "confidence": 0.9,
                "source": data.get("name")
            })

        # ---------- TABLE ----------
        elif ref in table_map:
            tb = table_map[ref]
            page_no = tb.get("prov", [{}])[0].get("page_no")

            # Prefer semantic flattening over fake structure
            cell_texts = []
            for cell in tb.get("cells", []):
                if isinstance(cell, dict):
                    val = cell.get("text")
                    if val:
                        cell_texts.append(val)

            table_text = " | ".join(cell_texts)

            if table_text.strip():
                chunks.append({
                    "chunk_id": uuid.uuid4().hex,
                    "text": table_text,
                    "page": page_no,
                    "modality": "table",
                    "docling_label": str(tb.get("label")),
                    "confidence": 0.5,  # conservative
                    "source": data.get("name")
                })

        # ---------- FIGURE (caption-first) ----------
        elif ref in picture_map:
            pic = picture_map[ref]
            page_no = pic.get("prov", [{}])[0].get("page_no")
            caption = pic.get("caption")

            if caption:
                chunks.append({
                    "chunk_id": uuid.uuid4().hex,
                    "text": caption,
                    "page": page_no,
                    "modality": "figure",
                    "docling_label": "figure_caption",
                    "confidence": 0.6,
                    "source": data.get("name")
                })

    return chunks


In [10]:
chunks = create_research_chunks(data_dict)

In [11]:
from collections import Counter

Counter(chunk["modality"] for chunk in chunks)


Counter({'text': 158})

In [70]:
from IPython.display import Markdown, display, JSON

#display(JSON(result.document.model_dump_json()))

In [81]:
type(json_data)

method

In [14]:
result.document.__dict__.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [1]:
result.document.tables[0].__dict__

NameError: name 'result' is not defined

In [ ]:
source = r"D:\3. Research Main Room\Publications\1 Current Projects\Data Science and Tourism Research - Conceptual Study\DATA ANALYSIS\Data_files\2024_Data\2024 - 15.pdf"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)

[INFO] 2026-02-08 12:44:06,779 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 12:44:06,780 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-08 12:44:06,798 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 12:44:06,800 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 12:44:06,971 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 12:44:06,973 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-08 12:44:06,976 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-08 12:44:06,976 [RapidOCR] main.py:50: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\r

In [ ]:
from IPython.display import Markdown, display

# display(Markdown(result.document.export_to_markdown()))

In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker()

doc_chunks = chunker.chunk(result.document)

# RAG-ready chunks with provenance
This cell builds retrieval payloads with page numbers, headings, modality, and source path using Docling's `HybridChunker`.

In [95]:
import json
from pathlib import Path
from docling.chunking import HybridChunker

# Use HybridChunker with token-aware splitting (keep <= 512 for MiniLM)
chunker = HybridChunker(
    tokenizer="sentence-transformers/all-MiniLM-L6-v2",
    max_tokens=480,
    merge_peers=True,
    )

chunk_iter = chunker.chunk(result.document)
rag_payloads = []

for i, chunk in enumerate(chunk_iter):
    page_numbers = set()
    headings = []
    modalities = set()

    if chunk.meta and chunk.meta.doc_items:
        for item in chunk.meta.doc_items:
            label = getattr(item, "label", None)
            if label:
                modalities.add(str(label))
            if hasattr(item, "prov") and item.prov:
                for p in item.prov:
                    page_numbers.add(p.page_no)

        # Capture headings if present
        if getattr(chunk.meta, "headings", None):
            headings = [str(h) for h in chunk.meta.headings]

    payload = {
        "id": f"chunk_{i}",
        "content": chunk.text,
        "metadata": {
            "source": source,
            "pages": sorted(page_numbers),
            "headings": headings,
            "modality": sorted(modalities) if modalities else ["text"],
        },
    }
    rag_payloads.append(payload)

print(f"Total chunks: {len(rag_payloads)}")
print("Sample:")
print(json.dumps(rag_payloads[0], indent=2)[:800])

# Optional: write JSONL for vector DB ingestion
output_path = Path("rag_payloads.jsonl")
with output_path.open("w", encoding="utf-8") as f:
    for item in rag_payloads:
        f.write(json.dumps(item, ensure_ascii=True) + "\n")
print(f"Saved to: {output_path.resolve()}")

Token indices sequence length is longer than the specified maximum sequence length for this model (639 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 88
Sample:
{
  "id": "chunk_0",
  "content": "Contents lists available at ScienceDirect",
  "metadata": {
    "source": "D:\\3. Research Main Room\\Publications\\1 Current Projects\\Data Science and Tourism Research - Conceptual Study\\DATA ANALYSIS\\Data_files\\2024_Data\\2024 - 1.pdf",
    "pages": [
      1
    ],
    "headings": [],
    "modality": [
      "text"
    ]
  }
}
Saved to: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\rag_payloads.jsonl


# Embedding-based retrieval (single PDF)
This builds embeddings for the chunks and retrieves the most relevant excerpts with page numbers.

In [99]:
# If needed, install: pip install sentence-transformers
import numpy as np
from sentence_transformers import SentenceTransformer

# 1) Build embeddings for all chunks
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [p["content"] for p in rag_payloads]
embeddings = model.encode(texts, normalize_embeddings=True)

# 2) Simple top-k retriever
def retrieve_top_k(query, k=5):
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(embeddings, q_emb)
    top_idx = np.argsort(-scores)[:k]
    results = []
    for idx in top_idx:
        item = rag_payloads[int(idx)]
        results.append({
            "score": float(scores[int(idx)]),
            "content": item["content"],
            "metadata": item["metadata"],
        })
    return results

# Example query for your judge agent
query = "Classify the paper: descriptive vs predictive vs exploratory. Evidence from methods and results."
top_hits = retrieve_top_k(query, k=5)
top_hits[:5]

[{'score': 0.3283633589744568,
  'content': 'journal homepage: www.elsevier.com/locate/jhtm',
  'metadata': {'source': 'D:\\3. Research Main Room\\Publications\\1 Current Projects\\Data Science and Tourism Research - Conceptual Study\\DATA ANALYSIS\\Data_files\\2024_Data\\2024 - 1.pdf',
   'pages': [1],
   'headings': ['Journal of Hospitality and Tourism Management'],
   'modality': ['text']}},
 {'score': 0.31222543120384216,
  'content': 'Axial coding regrouped the initial categories summarized by open coding. The coders continued to continuously compare and analyze the potential  implicit  structures  among  the  initial  categories,  uncovering new  concepts  and  categories  by  connecting  these  independent  categories. They explored the internal logical relationships between them, and classified and aggregated the initial categories based on these associations to obtain the main categories. Subsequently, five main categories were identified by merging the related concepts that r

In [102]:
# Local Ollama multi-agent workflow (extractor -> judge -> summarizer)
import json
import urllib.request
import urllib.error

OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "llama3.1:8b"

def ollama_chat(messages, model=OLLAMA_MODEL, url=OLLAMA_URL, temperature=0.2):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
        "format": "json",
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        out = json.loads(resp.read().decode("utf-8"))
    return out.get("message", {}).get("content", "")

def safe_json_loads(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}

def build_context_from_hits(hits, max_chars=6000):
    parts = []
    total = 0
    for h in hits:
        snippet = h.get("content", "")
        meta = h.get("metadata", {})
        pages = meta.get("pages") or meta.get("page_numbers") or []
        heading = meta.get("heading") or (meta.get("headings") or [""])[0]
        chunk_id = h.get("parent_id", "")
        block = (
            f"CHUNK_ID: {chunk_id}\n"
            f"PAGES: {pages}\n"
            f"HEADING: {heading}\n"
            f"TEXT: {snippet}\n"
        )
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts)

# Query used for retrieval (reuse if already defined)
if "query" not in globals():
    query = "Classify the paper: descriptive vs predictive vs exploratory. Evidence from methods and results."

# Use top_hits from the retrieval cell
context_text = build_context_from_hits(top_hits)

schema_hint = {
    "summary": "string",
    "key_points": ["string"],
    "citations": [{"text": "string", "page": 1, "chunk_id": "string"}],
    "confidence": 0.0,
}

system_msg = (
    "You are a careful research assistant. Return ONLY valid JSON that matches the schema. "
    "Do not add extra keys. If unsure, be explicit in summary."
)

extractor_prompt = (
    "Extract evidence from the context to answer the query. "
    "Cite the most relevant chunks with page numbers."
)

judge_prompt = (
    "Check the extraction for grounding and missing evidence. "
    "Fix any unsupported claims. Keep JSON valid."
)

summarizer_prompt = (
    "Provide a concise structured response based on the judged extraction."
)

extractor_messages = [
    {"role": "system", "content": system_msg},
    {"role": "user", "content": json.dumps({"schema": schema_hint, "query": query, "context": context_text}) + "\n" + extractor_prompt},
]
extractor_out = ollama_chat(extractor_messages)
extractor_json = safe_json_loads(extractor_out)

judge_messages = [
    {"role": "system", "content": system_msg},
    {"role": "user", "content": json.dumps({"schema": schema_hint, "query": query, "context": context_text, "extraction": extractor_json}) + "\n" + judge_prompt},
]
judge_out = ollama_chat(judge_messages)
judge_json = safe_json_loads(judge_out)

summarizer_messages = [
    {"role": "system", "content": system_msg},
    {"role": "user", "content": json.dumps({"schema": schema_hint, "query": query, "context": context_text, "judged": judge_json}) + "\n" + summarizer_prompt},
]
final_out = ollama_chat(summarizer_messages)
final_json = safe_json_loads(final_out)

final_json

{'summary': "The paper uses a mixed-methods approach to explore cultural resilience in China's cultural heritage sites.",
 'key_points': ['Grounded theory was used for qualitative coding analysis',
  'Back-to-back coding by 3 graduate students extracted critical information and labeled it',
  'Initial categories were identified through iterative comparisons and upward inductions',
  "The research team consisted of 2 professors, 2 doctoral students, and 2 master's students"],
 'citations': [{'text': 'Bui et al. (2020)',
   'page': None,
   'chunk_id': 'CHUNK_ID: PAGES: [5]'},
  {'text': 'Wang et al. (2023)',
   'page': None,
   'chunk_id': 'CHUNK_ID: PAGES: [6, 7]'},
  {'text': 'Wang et al. (2023)',
   'page': None,
   'chunk_id': 'CHUNK_ID: PAGES: [5]'}],
 'confidence': 0.8}